# ESN — Echo State Network | Dengue Weekly Forecasting by UF

Adaptive reservoir computing para previsão semanal de dengue.

**Quantificação de incerteza:** Seed Ensemble — N reservatórios com sementes distintas
geram N trajetórias de previsão. Os quantis dessas trajetórias formam os intervalos
preditivos de 50%, 80%, 90% e 95%, análogo ao Monte Carlo Dropout do MLP.

### 1. Imports & Configuration

In [1]:
import warnings, time, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from statsmodels.tsa.stattools import kpss
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import het_breuschpagan

warnings.filterwarnings("ignore")
np.random.seed(42)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE_DIR = Path(r"C:\myenv\env\Mestrado\2026_imdc_challenge")
DB_DIR   = BASE_DIR / "0_databases"
OUT_DIR  = BASE_DIR / "7_results_esn"
OUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------
TARGET    = "casos"
FREQ      = 52
PI_LEVELS = [0.50, 0.80, 0.90, 0.95]
N_SEEDS   = 50    # número de sementes para o Seed Ensemble (intervalos preditivos)
WASHOUT   = 50     # passos iniciais descartados no reservatório (transiente)

DROP_EXOG = {
    "pop_density_km2","BSh_koppen","As_koppen","rel_humid_med",
    "casos","date","year","epiweek","uf","uf_code",
    "regional_geocode","regional_health_name","macroregion_code","macroregion_name",
    "train_1","train_2","train_3","train_4",
    "target_1","target_2","target_3","target_4",
    "regional_health_area_km2", "precip_med", "rainy_days", "temp_med", "enso"
}

VALIDATIONS = [
    (1, "train_1", "target_1"),
    (2, "train_2", "target_2"),
    (3, "train_3", "target_3"),
    (4, "train_4", "target_4"),
]

# ---------------------------------------------------------------------------
# Hyperparameter search grid
# ESN é rápida (Ridge regression) → grid mais amplo que MLP
# ---------------------------------------------------------------------------
HP_GRID = {
    "n_reservoir"    : [100, 300, 500],
    "spectral_radius": [0.7, 0.95],
    "leaking_rate"   : [0.3, 0.7, 1.0],
    "sparsity"       : [0.05, 0.1],
    "ridge"          : [1e-4, 1e-1],
    "input_scaling"  : [0.5, 1.0],
}
# Para o grid search, usa N_SEEDS_SEARCH sementes por configuração (mais rápido)
# e N_SEEDS para o modelo final
N_SEEDS_SEARCH = 5


### 2. Data Loading & UF Aggregation

In [2]:
print("Carregando hierarch_forecast_lagged.parquet...")
hf_raw = pd.read_parquet(DB_DIR / "hierarch_forecast_lagged.parquet")
hf_raw = hf_raw[hf_raw["uf"] != "ES"]
hf_raw["date"]  = pd.to_datetime(hf_raw["date"])
hf_raw["casos"] = hf_raw["casos"].astype(float)

# Fill lagged climate columns
for col in [c for c in hf_raw.columns if "_lag" in c]:
    hf_raw[col] = hf_raw[col].ffill().bfill()

koppen_cols = [c for c in hf_raw.columns if c.endswith("_koppen")]
biome_cols  = [c for c in hf_raw.columns if c.endswith("_biome")]
dummy_cols  = koppen_cols + biome_cols

climate_cols = ["pressure_med", "thermal_range", "precip_med_lag11",
                "rainy_days_lag17", "temp_med_lag15", "enso_lag26"]
flag_cols    = ["train_1","train_2","train_3","train_4",
                "target_1","target_2","target_3","target_4"]

agg_dict = {
    "casos": "sum", "population": "sum",
    "uf_code": "first", "macroregion_code": "first", "macroregion_name": "first",
    "year": "first", "epiweek": "first",
    **{c: "mean" for c in climate_cols},
    **{c: "sum"  for c in dummy_cols},
    **{c: "first" for c in flag_cols},
}

hf = (hf_raw.groupby(["uf","date"], sort=True).agg(agg_dict).reset_index())
hf[dummy_cols] = hf[dummy_cols].clip(upper=1)
for c in flag_cols:
    hf[c] = hf[c].astype(bool)
hf = hf.sort_values(["uf","date"]).reset_index(drop=True)

exog_cols   = sorted(set(hf.columns) - DROP_EXOG
                     - {c for c in hf.columns if hf[c].dtype == object})
FEATURE_COLS = [TARGET] + [c for c in exog_cols if c != TARGET]
n_features   = len(FEATURE_COLS)
target_idx   = 0

uf_code_map = hf[["uf","uf_code"]].drop_duplicates("uf").set_index("uf")
ufs = sorted(hf["uf"].unique())

print(f"  UFs      : {len(ufs)}")
print(f"  Features ESN ({n_features}): {FEATURE_COLS[:5]} ...")
print(f"  Datas    : {hf['date'].min().date()} → {hf['date'].max().date()}")


Carregando hierarch_forecast_lagged.parquet...
  UFs      : 26
  Features ESN (21): ['casos', 'Af_koppen', 'Am_koppen', 'Amazônia_biome', 'Aw_koppen'] ...
  Datas    : 2010-01-03 → 2026-03-08


### 3. Echo State Network — Implementação e Funções Auxiliares

A ESN tem dois tipos de peso:
- **Fixos (não treinados):** W_in (entrada→reservatório) e W (reservatório→reservatório)
- **Treinados:** W_out (reservatório→saída), ajustados por Ridge Regression

A incerteza vem da **inicialização aleatória do reservatório** (seed). Rodando N_SEEDS modelos com sementes distintas, obtemos N_SEEDS trajetórias de previsão — os quantis dessas trajetórias formam os intervalos preditivos.

In [3]:
class EchoStateNetwork:
    """
    Echo State Network (ESN) para previsão de séries temporais multivariadas — MISO.

    Arquitetura:
      W_in  [N_res × n_inputs]   : pesos de entrada  (fixos, aleatórios)
      W     [N_res × N_res]      : pesos internos     (fixos, esparsos, escalonados)
      W_out [1     × (N_res+n_in)]: readout           (TREINADO via Ridge Regression)

    Equações de estado:
      x̃(t) = tanh(W_in·u(t) + W·x(t−1) + b + ε)
      x(t)  = (1−α)·x(t−1) + α·x̃(t)          [α = leaking rate]
      ŷ(t)  = W_out · [x(t); u(t)]
    """

    def __init__(self, n_reservoir=200, spectral_radius=0.9, sparsity=0.1,
                 leaking_rate=1.0, input_scaling=1.0, noise=1e-5,
                 ridge=1e-4, seed=None):
        self.n_reservoir     = n_reservoir
        self.spectral_radius = spectral_radius
        self.sparsity        = sparsity
        self.leaking_rate    = leaking_rate
        self.input_scaling   = input_scaling
        self.noise           = noise
        self.ridge           = ridge
        self.seed            = seed
        self.W_out = self.W = self.W_in = None

    def _init_weights(self, n_inputs):
        rng = np.random.RandomState(self.seed)
        self.W_in = rng.uniform(-1, 1, (self.n_reservoir, n_inputs)) * self.input_scaling
        W = rng.uniform(-1, 1, (self.n_reservoir, self.n_reservoir))
        mask = rng.rand(*W.shape) > self.sparsity
        W[mask] = 0.0
        eigs = np.linalg.eigvals(W)
        sr   = np.max(np.abs(eigs))
        if sr > 0:
            W *= self.spectral_radius / sr
        self.W     = W
        self.b_res = rng.uniform(-0.1, 0.1, self.n_reservoir)
        self._rng  = rng

    def _run_reservoir(self, X, washout=50, x_init=None):
        """
        Propaga X pelo reservatório, retorna (estados_após_washout, estado_final).
        x_init: estado inicial do reservatório (para forecast contínuo).
        """
        T  = X.shape[0]
        x  = np.zeros(self.n_reservoir) if x_init is None else x_init.copy()
        states = np.zeros((T, self.n_reservoir))
        rng = np.random.RandomState(self.seed)   # ruído reprodutível
        for t in range(T):
            u      = X[t]
            pre    = self.W_in @ u + self.W @ x + self.b_res
            pre   += self.noise * rng.randn(self.n_reservoir)
            x_new  = np.tanh(pre)
            x      = (1.0 - self.leaking_rate) * x + self.leaking_rate * x_new
            states[t] = x
        return states[washout:], x   # (T-washout, N_res), estado final

    def fit(self, X, y, washout=50):
        """Treina W_out via Ridge Regression."""
        if X.ndim == 1: X = X.reshape(-1, 1)
        if y.ndim == 1: y = y.reshape(-1, 1)
        self._init_weights(X.shape[1])
        self.washout_ = washout
        states, self._x_last = self._run_reservoir(X, washout)
        extended  = np.hstack([states, X[washout:]])
        reg = Ridge(alpha=self.ridge, fit_intercept=False)
        reg.fit(extended, y[washout:])
        self.W_out = reg.coef_
        self._n_ext = extended.shape[1]
        return self

    def predict_teacher(self, X, washout=0, x_init=None):
        """Predição com teacher forcing (X fornecido externamente)."""
        if X.ndim == 1: X = X.reshape(-1, 1)
        states, x_last = self._run_reservoir(X, washout, x_init)
        extended = np.hstack([states, X[washout:]])
        return (extended @ self.W_out.T).squeeze(), x_last

    def predict_recursive(self, x_seed_state, seed_input_row, h_total,
                          exog_fc, target_idx=0):
        """
        Previsão recursiva: prediz h_total passos usando apenas
        o estado do reservatório herdado do treino + exog futuros.

        x_seed_state : estado final do reservatório após o treino
        seed_input_row: última linha de input do treino (escalada)
        exog_fc      : (h_total, n_exog) — exógenas futuras (escaladas)
        """
        x = x_seed_state.copy()
        u = seed_input_row.copy()
        preds = []
        rng = np.random.RandomState(self.seed)

        for t in range(h_total):
            # Atualiza exógenas com os valores futuros (exclui target)
            u_next = u.copy()
            if exog_fc is not None:
                mask_exog = [i for i in range(len(u)) if i != target_idx]
                u_next[mask_exog] = exog_fc[t]

            # Passo do reservatório
            pre   = self.W_in @ u_next + self.W @ x + self.b_res
            pre  += self.noise * rng.randn(self.n_reservoir)
            x_new = np.tanh(pre)
            x     = (1.0 - self.leaking_rate) * x + self.leaking_rate * x_new

            # Predição
            ext   = np.concatenate([x, u_next])
            y_hat = float(self.W_out @ ext)
            preds.append(y_hat)

            # Feedback: atualiza target no input para próximo passo
            u_next[target_idx] = y_hat
            u = u_next

        return np.array(preds)


# ---------------------------------------------------------------------------
# Seed Ensemble: N_SEEDS instâncias com sementes distintas → intervalos
# ---------------------------------------------------------------------------
def esn_seed_ensemble(hp, X_train_sc, y_train_sc,
                      last_row_sc, exog_fc_sc,
                      h_total, target_idx,
                      n_seeds=N_SEEDS, washout=WASHOUT):
    """
    Treina N instâncias ESN com sementes distintas e faz previsão recursiva
    com cada uma. Retorna matriz (h_total, n_seeds) de trajetórias.
    """
    trajs = np.zeros((h_total, n_seeds))
    for s in range(n_seeds):
        esn = EchoStateNetwork(**hp, seed=s)
        esn.fit(X_train_sc, y_train_sc.ravel(), washout=washout)
        traj = esn.predict_recursive(
            x_seed_state  = esn._x_last,
            seed_input_row= last_row_sc,
            h_total       = h_total,
            exog_fc       = exog_fc_sc,
            target_idx    = target_idx,
        )
        trajs[:, s] = traj
    return trajs   # (h_total, n_seeds)


### 4. Metrics (WIS, MAE, RMSE, MAPE)

In [4]:
def wis(y_true, median, lower, upper, levels):
    K = len(levels)
    s = 0.5 * np.abs(y_true - median)
    for a in levels:
        IS = (upper[a] - lower[a]
              + (2/a) * np.maximum(0, lower[a] - y_true)
              + (2/a) * np.maximum(0, y_true - upper[a]))
        s += (a/2) * IS
    return float(np.mean(s) / (K + 0.5))


def compute_metrics(y_true, y_pred_med, lower, upper, label):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred_med, dtype=float)
    mask = ~(np.isnan(yt) | np.isnan(yp))
    yt, yp = yt[mask], yp[mask]
    mae  = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp)**2)))
    mape = float(np.nanmean(np.abs((yt - yp) / np.where(yt==0, np.nan, yt))) * 100)
    wis_val = wis(yt, yp,
                  {a: np.array(lower[a])[mask] for a in PI_LEVELS},
                  {a: np.array(upper[a])[mask] for a in PI_LEVELS},
                  PI_LEVELS)
    return {"model": label, "MAE": mae, "RMSE": rmse, "MAPE": mape, "WIS": wis_val}


def breusch_pagan_pval(y):
    x = np.hstack([np.ones((len(y),1)), np.arange(len(y)).reshape(-1,1)])
    res = OLS(y, x).fit()
    _, p, _, _ = het_breuschpagan(res.resid, x)
    return p

def n_diffs_kpss(y, max_d=2):
    for d in range(max_d+1):
        s = np.diff(y, n=d) if d > 0 else y
        try:
            _, p, _, _ = kpss(s, regression="c", nlags="auto")
            if p >= 0.05: return d
        except: return d
    return max_d


def save_forecast_plot(train_dates, train_vals, target_dates, target_vals,
                       fc_dates, fc_med, fc_lo, fc_hi, title, path):
    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(train_dates,  train_vals,  color="#2166ac", lw=0.7, label="Treino")
    ax.plot(target_dates, target_vals, color="black",   lw=1.2, label="Observado")
    pal = ["#f4a582","#d6604d","#b2182b","#67001f"]
    for i, a in enumerate(sorted(fc_lo.keys(), reverse=True)):
        ax.fill_between(fc_dates, fc_lo[a], fc_hi[a],
                        alpha=0.25, color=pal[i], label=f"IP {int(a*100)}%")
    ax.plot(fc_dates, fc_med, color="#d6604d", lw=1.2, ls="--", label="Mediana")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_ylabel("Casos")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.legend(fontsize=7, ncol=4)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)


### 5. Main Pipeline — Loop por UF e Validation Test

**Fluxo por validation:**
1. Breusch-Pagan → decidir log1p | KPSS → decidir diferenciação
2. MinMaxScaler ajustado **apenas** no treino
3. Grid search: para cada combinação de hiperparâmetros, treina N_SEEDS_SEARCH ESNs e avalia MSE médio no conjunto interno de validação (últimos 15%)
4. Fit final com os melhores HP + Seed Ensemble de N_SEEDS=100 sementes
5. Previsão recursiva: cada semente gera uma trajetória → quantis = IPs
6. Inversão das transformações → espaço original de casos

In [5]:
all_forecasts, all_metrics = [], []
txt_lines = []
t0_total  = time.time()

for i_uf, uf_label in enumerate(ufs):

    rd      = hf[hf["uf"] == uf_label].sort_values("date").reset_index(drop=True)
    uf_code = int(uf_code_map.loc[uf_label, "uf_code"])

    txt_lines.append(f"\n{'='*70}")
    txt_lines.append(f"UF: {uf_label}  (uf_code={uf_code})")
    txt_lines.append(f"{'='*70}")
    print(f"[{i_uf+1}/{len(ufs)}] UF: {uf_label}")

    # ── Transformações preliminares ───────────────────────────────────────
    y_full  = rd[TARGET].values.astype(float)
    bp_pval = breusch_pagan_pval(np.clip(y_full, 1e-3, None))
    use_log = bp_pval < 0.05
    y_tr_full = np.log1p(y_full) if use_log else y_full
    d_ord   = n_diffs_kpss(y_tr_full)
    inv     = (lambda x: np.expm1(x)) if use_log else (lambda x: np.array(x))

    txt_lines.append(f"Breusch-Pagan p={bp_pval:.4f} → {'log1p' if use_log else 'nível'}")
    txt_lines.append(f"KPSS d sugerido = {d_ord}")

    uf_out_dir = OUT_DIR / uf_label
    uf_out_dir.mkdir(exist_ok=True)

    # Exog futuras: média sazonal por semana do ano
    exog_hist = (rd.assign(w=rd["date"].dt.isocalendar().week.astype(int))
                   .groupby("w")[exog_cols].mean())

    for val_num, train_flag, target_flag in VALIDATIONS:

        txt_lines.append(f"\n  -- Validation {val_num} --")
        train_mask  = rd[train_flag].astype(bool)
        target_mask = rd[target_flag].astype(bool)

        if train_mask.sum() < FREQ or target_mask.sum() == 0:
            txt_lines.append("  [SKIP]"); continue

        tr  = rd[train_mask].sort_values("date")
        tgt = rd[target_mask].sort_values("date")

        y_tr_raw  = tr[TARGET].values.astype(float)
        y_tgt_raw = tgt[TARGET].values.astype(float)

        last_train  = tr["date"].max()
        first_tgt   = tgt["date"].min()
        gap         = int((first_tgt - last_train).days / 7)
        h_total     = gap + len(tgt)

        # ── V4: estende horizonte até EW40/2026 (2026-10-04) ─────────────────
        # O target_4 na base pode estar incompleto pois os dados ainda não foram
        # totalmente reportados. Este bloco garante que a previsão vai até
        # 2026-10-04 (EW40/2026), preenchendo y_tgt_raw com NaN para as semanas
        # futuras ainda sem dados (metrics já ignoram NaN via máscara interna).
        TARGET_V4_END = pd.Timestamp("2026-10-04")
        if val_num == 4:
            required_h = int((TARGET_V4_END - last_train).days / 7)
            if required_h > h_total:
                h_total = required_h
        tgt_dates_ext = pd.date_range(first_tgt, periods=h_total - gap, freq="W-SUN")
        tgt_date_map  = dict(zip(pd.to_datetime(tgt["date"].values), y_tgt_raw))
        y_tgt_raw     = np.array([tgt_date_map.get(d, np.nan) for d in tgt_dates_ext])

        txt_lines.append(f"  Treino : {tr['date'].min().date()} → {last_train.date()} ({len(tr)} sem)")
        txt_lines.append(f"  Gap    : {gap} semanas")
        txt_lines.append(f"  Target : {first_tgt.date()} → {tgt['date'].max().date()} ({len(tgt)} sem)")

        fc_dates = pd.date_range(last_train + pd.Timedelta(weeks=1),
                                 periods=h_total, freq="W-SUN")

        # ── Preparação dos dados ──────────────────────────────────────────
        tr_feat = tr[FEATURE_COLS].copy().astype(float)
        if use_log:
            tr_feat[TARGET] = np.log1p(tr_feat[TARGET].values)

        # Diferenciação do target (se necessário)
        last_log_val = tr_feat[TARGET].values[-1]
        if d_ord == 1:
            tr_feat[TARGET] = tr_feat[TARGET].diff().fillna(0)

        # MinMaxScaler ajustado no treino
        scaler = MinMaxScaler()
        X_train_sc = scaler.fit_transform(tr_feat.values)   # (T, n_features)
        y_train_sc = X_train_sc[:, target_idx]

        # Última linha do treino (para seed do forecast recursivo)
        last_row_sc = X_train_sc[-1].copy()

        # Exógenas futuras (escaladas) — excluindo target
        fc_df_idx = rd[rd["date"].isin(fc_dates)].set_index("date")
        fc_rows = []
        for d in fc_dates:
            if d in fc_df_idx.index:
                fc_rows.append(fc_df_idx.loc[d, exog_cols].values.astype(float))
            else:
                wk = min(int(d.isocalendar().week), 52)
                fc_rows.append(exog_hist.loc[wk].values
                               if wk in exog_hist.index else exog_hist.mean().values)
        exog_fc_raw = np.array(fc_rows, dtype=float)   # (h_total, n_exog)

        # Escala as exógenas futuras usando o scaler do treino
        # (cria array completo com zeros na coluna target, extrai exog)
        exog_idx = [i for i in range(n_features) if i != target_idx]
        dummy_full = np.zeros((h_total, n_features))
        dummy_full[:, exog_idx] = exog_fc_raw
        dummy_full_sc = scaler.transform(dummy_full)
        exog_fc_sc = dummy_full_sc[:, exog_idx]  # (h_total, n_exog)

        # ── Grid search de hiperparâmetros ────────────────────────────────
        print(f"    V{val_num} Grid Search...", end=" ", flush=True)
        t0 = time.time()

        # Divisão interna: últimos 15% do treino como validação
        n_int_val = max(int(len(X_train_sc) * 0.15), 2 * WASHOUT)
        n_tr_int  = len(X_train_sc) - n_int_val
        X_gs_tr   = X_train_sc[:n_tr_int]
        X_gs_val  = X_train_sc[n_tr_int:]
        y_gs_tr   = y_train_sc[:n_tr_int]
        y_gs_val  = y_train_sc[n_tr_int:]

        best_hp, best_mse_gs = None, np.inf

        hp_combos = list(itertools.product(
            HP_GRID["n_reservoir"],
            HP_GRID["spectral_radius"],
            HP_GRID["leaking_rate"],
            HP_GRID["sparsity"],
            HP_GRID["ridge"],
            HP_GRID["input_scaling"],
        ))

        for (nr, sr, lr, sp, rg, iscl) in hp_combos:
            hp_cand = dict(n_reservoir=nr, spectral_radius=sr, leaking_rate=lr,
                           sparsity=sp, ridge=rg, input_scaling=iscl, noise=1e-5)
            if n_tr_int <= WASHOUT + 5:
                continue

            # Avalia MSE médio sobre N_SEEDS_SEARCH sementes (teacher forcing no val)
            mses = []
            for s in range(N_SEEDS_SEARCH):
                try:
                    esn_c = EchoStateNetwork(**hp_cand, seed=s)
                    esn_c.fit(X_gs_tr, y_gs_tr, washout=WASHOUT)
                    pred_val, _ = esn_c.predict_teacher(
                        X_gs_val, washout=0, x_init=esn_c._x_last
                    )
                    mses.append(mean_squared_error(y_gs_val, pred_val))
                except Exception:
                    pass

            if mses:
                mse_avg = float(np.mean(mses))
                if mse_avg < best_mse_gs:
                    best_mse_gs = mse_avg
                    best_hp     = hp_cand.copy()

        if best_hp is None:
            best_hp = dict(n_reservoir=200, spectral_radius=0.9, leaking_rate=0.7,
                           sparsity=0.1, ridge=1e-2, input_scaling=1.0, noise=1e-5)
        print(f"done ({time.time()-t0:.0f}s) | n_res={best_hp['n_reservoir']} "
              f"sr={best_hp['spectral_radius']} lr={best_hp['leaking_rate']} "
              f"ridge={best_hp['ridge']}", flush=True)

        # ── Seed Ensemble: N_SEEDS modelos com HP ótimos ─────────────────
        print(f"    V{val_num} Seed Ensemble ({N_SEEDS} seeds)...", end=" ", flush=True)
        t0 = time.time()

        trajs_sc = esn_seed_ensemble(
            hp          = best_hp,
            X_train_sc  = X_train_sc,
            y_train_sc  = y_train_sc,
            last_row_sc = last_row_sc,
            exog_fc_sc  = exog_fc_sc,
            h_total     = h_total,
            target_idx  = target_idx,
            n_seeds     = N_SEEDS,
            washout     = WASHOUT,
        )
        # trajs_sc: (h_total, N_SEEDS) — em espaço escalado+diferenciado
        print(f"done ({time.time()-t0:.0f}s)", flush=True)

        # ── Inversão das transformações ───────────────────────────────────
        def inv_traj(traj_sc_col):
            """(h_total,) escalado → casos originais"""
            tmp = np.zeros((h_total, n_features))
            tmp[:, target_idx] = traj_sc_col
            unscaled_diff = scaler.inverse_transform(tmp)[:, target_idx]
            if d_ord == 1:
                vals = [last_log_val + unscaled_diff[0]]
                for i in range(1, h_total):
                    vals.append(vals[-1] + unscaled_diff[i])
                unscaled_log = np.array(vals)
            else:
                unscaled_log = unscaled_diff
            return np.clip(inv(unscaled_log), 0, None)

        trajs_orig = np.apply_along_axis(inv_traj, 0, trajs_sc)
        # trajs_orig: (h_total, N_SEEDS) — espaço original de casos

        # Mediana e intervalos a partir das N_SEEDS trajetórias
        fc_median = np.median(trajs_orig, axis=1)
        fc_lower  = {a: np.quantile(trajs_orig, (1-a)/2,   axis=1) for a in PI_LEVELS}
        fc_upper  = {a: np.quantile(trajs_orig, 1-(1-a)/2, axis=1) for a in PI_LEVELS}

        # ── Métricas (apenas período target) ─────────────────────────────
        fc_med_tgt = fc_median[gap:]
        fc_lo_tgt  = {a: fc_lower[a][gap:] for a in PI_LEVELS}
        fc_hi_tgt  = {a: fc_upper[a][gap:] for a in PI_LEVELS}

        metrics = compute_metrics(y_tgt_raw, fc_med_tgt, fc_lo_tgt, fc_hi_tgt, "ESN")
        print(f"    MAE={metrics['MAE']:.1f}  RMSE={metrics['RMSE']:.1f}  "
              f"MAPE={metrics['MAPE']:.1f}%  WIS={metrics['WIS']:.2f}")

        # ── Relatório TXT ─────────────────────────────────────────────────
        txt_lines.append(f"  ESN Hiperparâmetros (melhor configuração):")
        txt_lines.append(f"    n_reservoir     : {best_hp['n_reservoir']}")
        txt_lines.append(f"    spectral_radius : {best_hp['spectral_radius']}")
        txt_lines.append(f"    leaking_rate    : {best_hp['leaking_rate']}")
        txt_lines.append(f"    sparsity        : {best_hp['sparsity']}")
        txt_lines.append(f"    ridge           : {best_hp['ridge']}")
        txt_lines.append(f"    input_scaling   : {best_hp['input_scaling']}")
        txt_lines.append(f"    noise           : {best_hp['noise']}")
        txt_lines.append(f"    n_seeds_ensemble: {N_SEEDS}")
        txt_lines.append(f"    washout         : {WASHOUT}")
        txt_lines.append(f"    use_log         : {use_log}")
        txt_lines.append(f"    d_diff          : {d_ord}")
        txt_lines.append(f"    val_MSE (search): {best_mse_gs:.6f}")
        txt_lines.append(f"  Métricas:")
        txt_lines.append(f"    MAE={metrics['MAE']:.1f}  RMSE={metrics['RMSE']:.1f}  "
                         f"MAPE={metrics['MAPE']:.1f}%  WIS={metrics['WIS']:.2f}")

        # ── Gráfico forecast ──────────────────────────────────────────────
        save_forecast_plot(
            tr["date"].values, y_tr_raw,
            tgt_dates_ext, y_tgt_raw,
            fc_dates, fc_median, fc_lower, fc_upper,
            (f"ESN V{val_num} | {uf_label}  "
             f"(N_res={best_hp['n_reservoir']}, sr={best_hp['spectral_radius']}, "
             f"seeds={N_SEEDS}, WIS={metrics['WIS']:.2f})"),
            uf_out_dir / f"v{val_num}_esn_forecast.png",
        )

        # ── Dispersão das trajetórias (diagnóstico do ensemble) ───────────
        fig_ens, ax_ens = plt.subplots(figsize=(14,4))
        t_idx = np.arange(h_total)
        for s in range(min(30, N_SEEDS)):   # plota 30 trajetórias individuais
            ax_ens.plot(fc_dates, trajs_orig[:, s],
                        color="#4393c3", alpha=0.15, lw=0.5)
        ax_ens.plot(fc_dates, fc_median, color="#d6604d", lw=1.5, label="Mediana")
        ax_ens.plot(tgt_dates_ext, y_tgt_raw, color="black",
                    lw=1.2, label="Observado")
        ax_ens.set_title(f"Seed Ensemble V{val_num} | {uf_label} "
                         f"({N_SEEDS} trajetórias)", fontsize=9, fontweight="bold")
        ax_ens.legend(fontsize=7)
        ax_ens.spines[["top","right"]].set_visible(False)
        plt.tight_layout()
        fig_ens.savefig(uf_out_dir / f"v{val_num}_esn_ensemble.png",
                        dpi=120, bbox_inches="tight")
        plt.close(fig_ens)

        # ── Coleta resultados ─────────────────────────────────────────────
        all_metrics.append({
            "uf": uf_label, "uf_code": uf_code,
            "validation": val_num, "model": "ESN",
            "n_reservoir"    : best_hp["n_reservoir"],
            "spectral_radius": best_hp["spectral_radius"],
            "leaking_rate"   : best_hp["leaking_rate"],
            "sparsity"       : best_hp["sparsity"],
            "ridge"          : best_hp["ridge"],
            "input_scaling"  : best_hp["input_scaling"],
            "n_seeds"        : N_SEEDS,
            "washout"        : WASHOUT,
            "use_log"        : use_log,
            "d_diff"         : d_ord,
            "val_MSE_search" : best_mse_gs,
            **metrics,
        })
        for i, (d, obs) in enumerate(zip(tgt_dates_ext, y_tgt_raw)):
            row = {"uf": uf_label, "uf_code": uf_code,
                   "validation": val_num, "model": "ESN",
                   "date": d, "y_true": obs,
                   "median": fc_med_tgt[i]}
            for a in PI_LEVELS:
                row[f"lower_{int(a*100)}"] = fc_lo_tgt[a][i]
                row[f"upper_{int(a*100)}"] = fc_hi_tgt[a][i]
            all_forecasts.append(row)

print(f"\nPipeline ESN: {(time.time()-t0_total)/60:.1f} min total")


[1/26] UF: AC
    V1 Grid Search... done (74s) | n_res=100 sr=0.7 lr=0.3 ridge=0.0001
    V1 Seed Ensemble (50 seeds)... done (1s)
    MAE=66.6  RMSE=101.0  MAPE=54.4%  WIS=66.56
    V2 Grid Search... done (87s) | n_res=100 sr=0.7 lr=0.3 ridge=0.0001
    V2 Seed Ensemble (50 seeds)... done (1s)
    MAE=155.2  RMSE=281.8  MAPE=62.4%  WIS=155.23
    V3 Grid Search... done (91s) | n_res=100 sr=0.7 lr=0.3 ridge=0.0001
    V3 Seed Ensemble (50 seeds)... done (1s)
    MAE=188.0  RMSE=271.2  MAPE=69.6%  WIS=187.98
    V4 Grid Search... done (139s) | n_res=100 sr=0.7 lr=0.3 ridge=0.0001
    V4 Seed Ensemble (50 seeds)... done (1s)
    MAE=28.6  RMSE=37.2  MAPE=44.4%  WIS=28.30
[2/26] UF: AL
    V1 Grid Search... done (126s) | n_res=100 sr=0.7 lr=0.3 ridge=0.0001
    V1 Seed Ensemble (50 seeds)... done (1s)
    MAE=736.0  RMSE=790.0  MAPE=1202.5%  WIS=734.93
    V2 Grid Search... done (182s) | n_res=100 sr=0.7 lr=0.3 ridge=0.0001
    V2 Seed Ensemble (50 seeds)... done (3s)
    MAE=357.7  RMSE=

### 6. Export Results

In [6]:
print("Exportando resultados...")
df_metrics   = pd.DataFrame(all_metrics)
df_forecasts = pd.DataFrame(all_forecasts)
df_metrics.to_parquet(OUT_DIR / "metrics_esn.parquet",    index=False)
df_forecasts.to_parquet(OUT_DIR / "forecasts_esn.parquet", index=False)

with open(OUT_DIR / "relatorio_esn.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(txt_lines))

print(f"  metrics_esn.parquet    → {len(df_metrics):,} linhas")
print(f"  forecasts_esn.parquet  → {len(df_forecasts):,} linhas")

print("\nResumo médio por Validation Test:")
print(df_metrics.groupby("validation")[["MAE","RMSE","MAPE","WIS"]].mean().round(2).to_string())

print("\nDistribuição dos melhores hiperparâmetros:")
for col in ["n_reservoir","spectral_radius","leaking_rate","ridge"]:
    print(f"  {col}: {df_metrics[col].value_counts().to_dict()}")


Exportando resultados...
  metrics_esn.parquet    → 104 linhas
  forecasts_esn.parquet  → 5,408 linhas

Resumo médio por Validation Test:
                     MAE          RMSE          MAPE           WIS
validation                                                        
1           8.480988e+17  3.528662e+18  2.704828e+19  8.477764e+17
2           1.131327e+13  4.079601e+13  1.110082e+14  1.113545e+13
3           1.356630e+03  1.792670e+03  2.677700e+02  1.355900e+03
4           2.980353e+04  5.516809e+04  9.873028e+05  2.991889e+04

Distribuição dos melhores hiperparâmetros:
  n_reservoir: {100: 104}
  spectral_radius: {0.7: 66, 0.95: 38}
  leaking_rate: {0.3: 97, 1.0: 7}
  ridge: {0.0001: 104}


### 7. Forecast Mosaic by UF

In [7]:
uf_dir = OUT_DIR / "plots_uf"
uf_dir.mkdir(exist_ok=True)
ufs_sorted = sorted(df_forecasts["uf"].unique())
N = len(ufs_sorted); N_COLS = 6; N_ROWS = int(np.ceil(N/N_COLS))
pal = ["#f4a582","#d6604d","#b2182b","#67001f"]

for val_num in [1,2,3,4]:
    df_v = df_forecasts[df_forecasts["validation"]==val_num]
    if df_v.empty: continue
    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(24, N_ROWS*3.5),
                              constrained_layout=True)
    axes_flat = axes.flatten()
    for i, uf in enumerate(ufs_sorted):
        ax  = axes_flat[i]
        sub = df_v[df_v["uf"]==uf].sort_values("date")
        if sub.empty: ax.set_visible(False); continue
        ax.plot(sub["date"], sub["y_true"], color="black", lw=1.0, label="Observado")
        for j, a in enumerate(sorted(PI_LEVELS, reverse=True)):
            ax.fill_between(sub["date"], sub[f"lower_{int(a*100)}"],
                            sub[f"upper_{int(a*100)}"], alpha=0.2, color=pal[j])
        ax.plot(sub["date"], sub["median"], color="#d6604d", lw=1.0, ls="--")
        ax.set_title(uf, fontsize=8, fontweight="bold")
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.tick_params(axis="x", labelsize=6, rotation=45)
        ax.tick_params(axis="y", labelsize=6)
        ax.spines[["top","right"]].set_visible(False)
    for j in range(N, len(axes_flat)): axes_flat[j].set_visible(False)
    fig.suptitle(f"Validation {val_num} — ESN | Seed Ensemble ({N_SEEDS} seeds) | Casos por UF",
                 fontsize=11, fontweight="bold")
    fig.savefig(uf_dir / f"v{val_num}_esn_uf_mosaic.png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: v{val_num}_esn_uf_mosaic.png")


  Salvo: v1_esn_uf_mosaic.png
  Salvo: v2_esn_uf_mosaic.png
  Salvo: v3_esn_uf_mosaic.png
  Salvo: v4_esn_uf_mosaic.png


### 8. WIS Heatmap — ESN × UF × Validation Test

In [8]:
for val_num in [1,2,3,4]:
    sub   = df_metrics[df_metrics["validation"]==val_num][["uf","WIS"]].copy()
    if sub.empty: continue
    pivot = sub.set_index("uf")["WIS"].to_frame().T
    pivot.index = ["ESN"]
    states = pivot.columns.tolist()

    fig, ax = plt.subplots(figsize=(max(12, len(states)*0.75), 2.5))
    im = ax.imshow(pivot.values, cmap="Reds",
                   vmin=np.nanpercentile(pivot.values,5),
                   vmax=np.nanpercentile(pivot.values,95), aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="WIS")
    ax.set_xticks(range(len(states))); ax.set_yticks([0])
    ax.set_xticklabels(states, fontsize=8)
    ax.set_yticklabels(["ESN"], fontsize=9)
    ax.set_xlabel("State", fontsize=9)
    vmax_ann = np.nanpercentile(pivot.values, 80)
    for j, uf in enumerate(states):
        v = pivot.loc["ESN", uf]
        if np.isnan(v): continue
        ax.text(j, 0, f"{v:.1f}", ha="center", va="center",
                fontsize=7, color="white" if v > vmax_ann else "black")
    ax.set_title(f"Validation test - {val_num} (WIS — ESN Seed Ensemble)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"heatmap_wis_esn_v{val_num}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: heatmap_wis_esn_v{val_num}.png")

print("\n✓ Concluído.")


  Salvo: heatmap_wis_esn_v1.png
  Salvo: heatmap_wis_esn_v2.png
  Salvo: heatmap_wis_esn_v3.png
  Salvo: heatmap_wis_esn_v4.png

✓ Concluído.
